In [5]:
import nbformat
import base64
import os

# ================= CONFIG =================


NOTEBOOK_MAP = {
    "main_1d_resnet_typ1_selfeeg.ipynb": "main_1d_resnet_typ1_selfeeg.ipynb",
    "main_1d_resnet_typ2_selfeeg.ipynb": "main_1d_resnet_typ2_selfeeg.ipynb"

}

FINAL_DIR = "result_and_plot_of_1d_resnet_selfeeg"
MAX_IMAGES = 5
EPOCH_KEY = "Epoch [1/25] | Train Loss:"
KEYWORDS = ["criterion =", "optimizer =", "scheduler =","model ="]

# =========================================

os.makedirs(FINAL_ROOT, exist_ok=True)


def process_notebook(src_nb, dst_name):
    if not os.path.exists(src_nb):
        print(f"⚠️  Skipped (not found): {src_nb}")
        return

    out_dir = os.path.join(FINAL_ROOT, dst_name.replace(".ipynb", ""))
    os.makedirs(out_dir, exist_ok=True)

    md_path = os.path.join(out_dir, "extracted.md")

    nb = nbformat.read(src_nb, as_version=4)

    # -------- A) Collect first 5 images --------
    images = []
    image_texts = []

    for cell in nb.cells:
        if len(images) >= MAX_IMAGES:
            break
        if cell.cell_type != "code":
            continue

        accumulated_text = ""

        for output in cell.get("outputs", []):
            if output.output_type == "stream":
                accumulated_text += output.get("text", "")

            elif output.output_type in ("display_data", "execute_result"):
                data = output.get("data", {})
                if "image/png" in data:
                    images.append(data["image/png"])
                    image_texts.append(accumulated_text.strip())
                    accumulated_text = ""

                    if len(images) >= MAX_IMAGES:
                        break

    # -------- B) Find FIRST Epoch cell --------
    training_output = ""
    training_config = []

    for cell in nb.cells:
        if cell.cell_type != "code":
            continue

        output_text = ""
        for output in cell.get("outputs", []):
            if output.output_type == "stream":
                output_text += output.get("text", "")

        if EPOCH_KEY not in output_text:
            continue

        training_output = output_text

        source_lines = cell.source.splitlines()
        capture = False
        balance = 0

        for line in source_lines:
            stripped = line.strip()

            if any(k in stripped for k in KEYWORDS):
                capture = True
                training_config.append(line)
                balance = stripped.count("(") - stripped.count(")")
                if balance <= 0:
                    capture = False
                continue

            if capture:
                training_config.append(line)
                balance += stripped.count("(") - stripped.count(")")
                if balance <= 0:
                    capture = False

        break

    # -------- Save images --------
    image_files = []
    for i, img_b64 in enumerate(images):
        img_path = os.path.join(out_dir, f"image_{i+1}.png")
        with open(img_path, "wb") as f:
            f.write(base64.b64decode(img_b64))
        image_files.append(img_path)

    # -------- Write Markdown --------
    with open(md_path, "w", encoding="utf-8") as md:
        md.write(f"# Extracted Notebook Data\n\n")
        md.write(f"**Source:** `{src_nb}`\n\n")
        md.write(f"**Renamed As:** `{dst_name}`\n\n")

        md.write("## 🖼 First Five Images\n\n")
        for i, img in enumerate(image_files):
            md.write(f"### Image {i+1}\n\n")
            if i < len(image_texts) and image_texts[i]:
                md.write("```text\n")
                md.write(image_texts[i])
                md.write("\n```\n\n")
            md.write(f"![image {i+1}]({os.path.basename(img)})\n\n")

        md.write("## 🔧 Training Configuration\n\n```python\n")
        md.write("\n".join(training_config) if training_config else "# Not found")
        md.write("\n```\n\n")

        md.write("## 📊 Training Output\n\n```text\n")
        md.write(training_output if training_output else "# Not found")
        md.write("\n```\n")





    print(f"✅ Done: {src_nb} → {out_dir}")


# ============ RUN LOOP ============
for src, dst in NOTEBOOK_MAP.items():
    process_notebook(src, dst)

print("\n🎯 All notebooks processed.")


✅ Done: main_1d_resnet_typ1_selfeeg.ipynb → result_and_plot_of_1d_resnet_selfeeg/main_1d_resnet_typ1_selfeeg
✅ Done: main_1d_resnet_typ2_selfeeg.ipynb → result_and_plot_of_1d_resnet_selfeeg/main_1d_resnet_typ2_selfeeg

🎯 All notebooks processed.


In [4]:
os.getcwd()

'/home/karansingh/Documents/summer term/ECG_ML/MAIN'

In [6]:
import nbformat
import os
import re
import pandas as pd

# ================= CONFIG =================


NOTEBOOK_MAP = {
    "main_1d_resnet_typ1_selfeeg.ipynb": "main_1d_resnet_typ1_selfeeg.ipynb",
    "main_1d_resnet_typ2_selfeeg.ipynb": "main_1d_resnet_typ2_selfeeg.ipynb"

}

FINAL_DIR = "result_and_plot_of_1d_resnet_selfeeg"
os.makedirs(FINAL_DIR, exist_ok=True)

MAX_EPOCH = 25
PATIENCE = 2

# =========================================


def one_line(s):
    """Make string Markdown-table safe and preserve key details"""
    if not s:
        return "—"
    cleaned = " ".join(s.replace("\n", " ").split())
    if len(cleaned) > 200:
        cleaned = cleaned[:197] + "..."
    return cleaned


def find_training_cell(nb):
    """Find the cell that contains the training loop (with output)"""
    for idx, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue
        
        # Check if this cell has training output
        has_output = False
        for out in cell.get("outputs", []):
            if out.output_type == "stream":
                text = out.get("text", "")
                if "Training Run Started:" in text or "Epoch [1/" in text:
                    has_output = True
                    break
        
        if has_output:
            return idx, cell.source
    
    return None, None


def extract_single_config(source, pattern):
    """Extract a single configuration line with multi-line support"""
    lines = source.splitlines()
    
    for i, line in enumerate(lines):
        # Look for the pattern with assignment
        if pattern in line and "=" in line:
            # Skip comments and variable assignments that aren't the config
            stripped = line.strip()
            if stripped.startswith("#"):
                continue
            
            # Found the line, now capture until balanced parentheses
            result = [line]
            balance = line.count("(") - line.count(")")
            
            # Simple assignment (no parentheses or balanced already)
            if balance <= 0:
                return line.strip()
            
            # Multi-line - need more lines
            j = i + 1
            while j < len(lines) and balance > 0:
                next_line = lines[j]
                result.append(next_line)
                balance += next_line.count("(") - next_line.count(")")
                j += 1
                
            return "\n".join(result).strip()
    
    return None


def extract_config(nb):
    """Extract configuration from the training loop cell itself"""
    found = {}
    
    # Find the training cell
    training_idx, training_source = find_training_cell(nb)
    
    if training_source is None:
        print("  ⚠️ Could not find training cell with output")
        return found
    
    print(f"  ✓ Found training cell at index {training_idx}")
    
    # Extract each config component from the training cell source
    model_cfg = extract_single_config(training_source, "model =")
    criterion_cfg = extract_single_config(training_source, "criterion =")
    optimizer_cfg = extract_single_config(training_source, "optimizer =")
    scheduler_cfg = extract_single_config(training_source, "scheduler =")
    
    if model_cfg:
        found["model ="] = model_cfg
    if criterion_cfg:
        found["criterion ="] = criterion_cfg
    if optimizer_cfg:
        found["optimizer ="] = optimizer_cfg
    if scheduler_cfg:
        found["scheduler ="] = scheduler_cfg
    
    print(f"  ✓ Extracted {len(found)}/4 config items")
    
    return found


def extract_training_log(nb):
    text = ""
    for cell in nb.cells:
        if cell.cell_type != "code":
            continue
        for out in cell.get("outputs", []):
            if out.output_type == "stream":
                text += out.get("text", "")
    return text


def parse_training(text):
    epoch_re = re.compile(
        r"Epoch \[(\d+)/\d+\] \| "
        r"Train Loss: ([\d.]+), Val Loss: ([\d.]+), "
        r"Train Acc: ([\d.]+)%, Val Acc: ([\d.]+)%, "
        r"LR: ([\d.eE+-]+)"
    )

    epochs = []
    for m in epoch_re.finditer(text):
        epochs.append({
            "epoch": int(m.group(1)),
            "train_loss": float(m.group(2)),
            "val_loss": float(m.group(3)),
            "val_acc": float(m.group(5)),
            "lr": float(m.group(6)),
        })

    if not epochs:
        return None

    best = max(epochs, key=lambda x: x["val_acc"])

    stop_match = re.search(r"Early stopping at epoch (\d+)", text)
    stop_epoch = int(stop_match.group(1)) if stop_match else MAX_EPOCH

    test_match = re.search(r"Test Accuracy:\s*([\d.]+)%", text)
    test_acc = float(test_match.group(1)) if test_match else None

    return {
        "best_epoch": best["epoch"],
        "best_val_acc": best["val_acc"],
        "train_loss_best": best["train_loss"],
        "val_loss_best": best["val_loss"],
        "init_lr": epochs[0]["lr"],
        "early_stop_epoch": stop_epoch,
        "test_acc": test_acc,
    }


# ================= RUN =================

config_rows = []
result_rows = []

for src, dst in NOTEBOOK_MAP.items():
    if not os.path.exists(src):
        print(f"⚠️ Missing: {src}")
        continue

    nb = nbformat.read(src, as_version=4)
    name = dst.replace(".ipynb", "")

    print(f"\n📓 {src} -> {name}")
    
    cfg = extract_config(nb)
    log = extract_training_log(nb)
    stats = parse_training(log)
    
    # Debug output - show what was found
    for key in ["model =", "criterion =", "optimizer =", "scheduler ="]:
        if key in cfg:
            preview = cfg[key][:80].replace("\n", " ")
            print(f"  {key}: {preview}...")

    config_rows.append([
        name,
        one_line(cfg.get("model =", "—")),
        one_line(cfg.get("criterion =", "—")),
        one_line(cfg.get("optimizer =", "—")),
        one_line(cfg.get("scheduler =", "—")),
        stats["init_lr"] if stats else "—",
    ])

    if stats:
        result_rows.append([
            name,
            MAX_EPOCH,
            "Yes",
            PATIENCE,
            stats["early_stop_epoch"],
            stats["best_epoch"],
            f"{stats['best_val_acc']:.2f}",
            f"{stats['train_loss_best']:.4f}",
            f"{stats['val_loss_best']:.4f}",
            f"{stats['test_acc']:.2f}" if stats["test_acc"] is not None else "—",
        ])


# ================= WRITE MARKDOWN =================

md_path = os.path.join(FINAL_DIR, "summary.md")

with open(md_path, "w", encoding="utf-8") as f:

    f.write("# 🔧 Training Configuration\n\n")
    f.write("| Notebook | Model | Criterion | Optimizer | Scheduler | Initial LR |\n")
    f.write("|---|---|---|---|---|---|\n")
    for r in config_rows:
        f.write("| " + " | ".join(map(str, r)) + " |\n")

    f.write("\n---\n\n")
    f.write("## 📊 Training Output Summary\n\n")
    f.write(
        "**Note:** Early stopping was enabled (patience = 2, max epoch = 25). "
        "All reported training and validation losses correspond to the epoch with the "
        "**highest validation accuracy**, not the final stopped epoch.\n\n"
    )

    f.write("| Notebook | Max Epoch | Early Stop | Patience | Stop Epoch | Best Epoch | "
            "Best Val Acc (%) | Train Loss | Val Loss | Test Acc (%) |\n")
    f.write("|---|---|---|---|---|---|---|---|---|---|\n")

    for r in result_rows:
        f.write("| " + " | ".join(map(str, r)) + " |\n")

print(f"\n✅ DONE → {md_path}")


# ================= WRITE EXCEL =================

xlsx_path = os.path.join(FINAL_DIR, "summary.xlsx")

# Training Configuration sheet
df_config = pd.DataFrame(
    config_rows,
    columns=[
        "Notebook",
        "Model",
        "Criterion",
        "Optimizer",
        "Scheduler",
        "Initial LR",
    ]
)

# Training Output Summary sheet
df_results = pd.DataFrame(
    result_rows,
    columns=[
        "Notebook",
        "Max Epoch",
        "Early Stop",
        "Patience",
        "Stop Epoch",
        "Best Epoch",
        "Best Val Acc (%)",
        "Train Loss (Best)",
        "Val Loss (Best)",
        "Test Acc (%)",
    ]
)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df_config.to_excel(writer, sheet_name="Training_Configuration", index=False)
    df_results.to_excel(writer, sheet_name="Training_Output_Summary", index=False)

print(f"📊 Excel saved → {xlsx_path}")



📓 main_1d_resnet_typ1_selfeeg.ipynb -> main_1d_resnet_typ1_selfeeg
  ✓ Found training cell at index 33
  ✓ Extracted 4/4 config items
  model =: model = ResNet1D(nb_classes=3, Chans=12, Samples=5000, Layers=[2,2,2,2], inplane...
  criterion =: criterion = nn.CrossEntropyLoss()...
  optimizer =: optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)...
  scheduler =: scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(         optimizer, mode=...

📓 main_1d_resnet_typ2_selfeeg.ipynb -> main_1d_resnet_typ2_selfeeg
  ✓ Found training cell at index 33
  ✓ Extracted 4/4 config items
  model =: model = ResNet1D(nb_classes=3, Chans=12, Samples=5000, Layers=[2,2,2,2], inplane...
  criterion =: criterion = nn.CrossEntropyLoss()...
  optimizer =: optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)...
  scheduler =: scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(         optimizer, mode=...

✅ DONE → result_and_plot_of_1d_resnet_selfeeg/s

In [5]:
import nbformat
import os
import re

# ================= CONFIG =================


NOTEBOOK_MAP = {
    "main_1d_resnet_typ1.ipynb": "1d_resnet_typ1.ipynb",
    "main_1d_resnet_typ2.ipynb": "1d_resnet_typ2.ipynb",

}

FINAL_DIR = "result_and_plot_of_1d_resnet"
os.makedirs(FINAL_DIR, exist_ok=True)

MAX_EPOCH = 25
PATIENCE = 2

# =========================================


def one_line(s):
    """Make string Markdown-table safe and preserve key details"""
    if not s:
        return "—"
    cleaned = " ".join(s.replace("\n", " ").split())
    if len(cleaned) > 200:
        cleaned = cleaned[:197] + "..."
    return cleaned


def find_training_cell(nb):
    """Find the cell that contains the training loop (with output)"""
    for idx, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue
        
        # Check if this cell has training output
        has_output = False
        for out in cell.get("outputs", []):
            if out.output_type == "stream":
                text = out.get("text", "")
                if "Training Run Started:" in text or "Epoch [1/" in text:
                    has_output = True
                    break
        
        if has_output:
            return idx, cell.source
    
    return None, None


def extract_single_config(source, pattern):
    """Extract a single configuration line with multi-line support"""
    lines = source.splitlines()
    
    for i, line in enumerate(lines):
        # Look for the pattern with assignment
        if pattern in line and "=" in line:
            # Skip comments and variable assignments that aren't the config
            stripped = line.strip()
            if stripped.startswith("#"):
                continue
            
            # Found the line, now capture until balanced parentheses
            result = [line]
            balance = line.count("(") - line.count(")")
            
            # Simple assignment (no parentheses or balanced already)
            if balance <= 0:
                return line.strip()
            
            # Multi-line - need more lines
            j = i + 1
            while j < len(lines) and balance > 0:
                next_line = lines[j]
                result.append(next_line)
                balance += next_line.count("(") - next_line.count(")")
                j += 1
                
            return "\n".join(result).strip()
    
    return None


def extract_config(nb):
    """Extract configuration from the training loop cell itself"""
    found = {}
    
    # Find the training cell
    training_idx, training_source = find_training_cell(nb)
    
    if training_source is None:
        print("  ⚠️ Could not find training cell with output")
        return found
    
    print(f"  ✓ Found training cell at index {training_idx}")
    
    # Extract each config component from the training cell source
    model_cfg = extract_single_config(training_source, "model =")
    criterion_cfg = extract_single_config(training_source, "criterion =")
    optimizer_cfg = extract_single_config(training_source, "optimizer =")
    scheduler_cfg = extract_single_config(training_source, "scheduler =")
    
    if model_cfg:
        found["model ="] = model_cfg
    if criterion_cfg:
        found["criterion ="] = criterion_cfg
    if optimizer_cfg:
        found["optimizer ="] = optimizer_cfg
    if scheduler_cfg:
        found["scheduler ="] = scheduler_cfg
    
    print(f"  ✓ Extracted {len(found)}/4 config items")
    
    return found


def extract_training_log(nb):
    text = ""
    for cell in nb.cells:
        if cell.cell_type != "code":
            continue
        for out in cell.get("outputs", []):
            if out.output_type == "stream":
                text += out.get("text", "")
    return text


def parse_training(text):
    epoch_re = re.compile(
        r"Epoch \[(\d+)/\d+\] \| "
        r"Train Loss: ([\d.]+), Val Loss: ([\d.]+), "
        r"Train Acc: ([\d.]+)%, Val Acc: ([\d.]+)%, "
        r"LR: ([\d.eE+-]+)"
    )

    epochs = []
    for m in epoch_re.finditer(text):
        epochs.append({
            "epoch": int(m.group(1)),
            "train_loss": float(m.group(2)),
            "val_loss": float(m.group(3)),
            "val_acc": float(m.group(5)),
            "lr": float(m.group(6)),
        })

    if not epochs:
        return None

    best = max(epochs, key=lambda x: x["val_acc"])

    stop_match = re.search(r"Early stopping at epoch (\d+)", text)
    stop_epoch = int(stop_match.group(1)) if stop_match else MAX_EPOCH

    test_match = re.search(r"Test Accuracy:\s*([\d.]+)%", text)
    test_acc = float(test_match.group(1)) if test_match else None

    # Extract training time
    time_match = re.search(r"Total Training Time:\s*([\d.]+)s", text)
    training_time = float(time_match.group(1)) if time_match else None

    return {
        "best_epoch": best["epoch"],
        "best_val_acc": best["val_acc"],
        "train_loss_best": best["train_loss"],
        "val_loss_best": best["val_loss"],
        "init_lr": epochs[0]["lr"],
        "early_stop_epoch": stop_epoch,
        "test_acc": test_acc,
        "training_time": training_time,
    }


# ================= RUN =================

config_rows = []
result_rows = []

for src, dst in NOTEBOOK_MAP.items():
    if not os.path.exists(src):
        print(f"⚠️ Missing: {src}")
        continue

    nb = nbformat.read(src, as_version=4)
    name = dst.replace(".ipynb", "")

    print(f"\n📓 {src} -> {name}")
    
    cfg = extract_config(nb)
    log = extract_training_log(nb)
    stats = parse_training(log)
    
    # Debug output - show what was found
    for key in ["model =", "criterion =", "optimizer =", "scheduler ="]:
        if key in cfg:
            preview = cfg[key][:80].replace("\n", " ")
            print(f"  {key}: {preview}...")

    config_rows.append([
        src,  # Real notebook name
        name,  # Display name
        one_line(cfg.get("model =", "—")),
        one_line(cfg.get("criterion =", "—")),
        one_line(cfg.get("optimizer =", "—")),
        one_line(cfg.get("scheduler =", "—")),
        stats["init_lr"] if stats else "—",
    ])

    if stats:
        training_time_str = f"{stats['training_time']:.2f}s" if stats["training_time"] is not None else "—"
        result_rows.append([
            src,  # Real notebook name
            name,  # Display name
            MAX_EPOCH,
            "Yes",
            PATIENCE,
            stats["early_stop_epoch"],
            stats["best_epoch"],
            f"{stats['best_val_acc']:.2f}",
            f"{stats['train_loss_best']:.4f}",
            f"{stats['val_loss_best']:.4f}",
            f"{stats['test_acc']:.2f}" if stats["test_acc"] is not None else "—",
            training_time_str,
        ])


# ================= WRITE MARKDOWN =================

md_path = os.path.join(FINAL_DIR, "summary.md")

with open(md_path, "w", encoding="utf-8") as f:

    f.write("# 🔧 Training Configuration\n\n")
    f.write("| Real Notebook | Display Name | Model | Criterion | Optimizer | Scheduler | Initial LR |\n")
    f.write("|---|---|---|---|---|---|---|\n")
    for r in config_rows:
        f.write("| " + " | ".join(map(str, r)) + " |\n")

    f.write("\n---\n\n")
    f.write("## 📊 Training Output Summary\n\n")
    f.write(
        "**Note:** Early stopping was enabled (patience = 2, max epoch = 25). "
        "All reported training and validation losses correspond to the epoch with the "
        "**highest validation accuracy**, not the final stopped epoch.\n\n"
    )

    f.write("| Real Notebook | Display Name | Max Epoch | Early Stop | Patience | Stop Epoch | Best Epoch | "
            "Best Val Acc (%) | Train Loss | Val Loss | Test Acc (%) | Training Time |\n")
    f.write("|---|---|---|---|---|---|---|---|---|---|---|---|\n")

    for r in result_rows:
        f.write("| " + " | ".join(map(str, r)) + " |\n")

print(f"\n✅ DONE → {md_path}")

import pandas as pd
config_columns = [
    "Real Notebook", "Display Name", "Model",
    "Criterion", "Optimizer", "Scheduler", "Initial LR"
]

result_columns = [
    "Real Notebook", "Display Name", "Max Epoch", "Early Stop",
    "Patience", "Stop Epoch", "Best Epoch",
    "Best Val Acc (%)", "Train Loss",
    "Val Loss", "Test Acc (%)", "Training Time"
]
df_config = pd.DataFrame(config_rows, columns=config_columns)
df_results = pd.DataFrame(result_rows, columns=result_columns)
excel_path = os.path.join(FINAL_DIR, "summary.xlsx")

with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    df_config.to_excel(writer, sheet_name="Training_Config", index=False)
    df_results.to_excel(writer, sheet_name="Training_Results", index=False)

print(f"📊 Excel summary saved → {excel_path}")



📓 main_1d_resnet_typ1.ipynb -> 1d_resnet_typ1
  ✓ Found training cell at index 37
  ✓ Extracted 4/4 config items
  model =: model = BenchmarkECGResNet1D(in_channels=12, num_classes=3).to(device)...
  criterion =: criterion = nn.CrossEntropyLoss()...
  optimizer =: optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)...
  scheduler =: scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(         optimizer, mode=...

📓 main_1d_resnet_typ2.ipynb -> 1d_resnet_typ2
  ✓ Found training cell at index 33
  ✓ Extracted 4/4 config items
  model =: model = BenchmarkECGResNet1D(in_channels=12, num_classes=3).to(device)...
  criterion =: criterion = nn.CrossEntropyLoss()...
  optimizer =: optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)...
  scheduler =: scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(         optimizer, mode=...

✅ DONE → result_and_plot_of_1d_resnet/summary.md
📊 Excel summary saved → result_and_plot_of_1d_resnet/summary

In [10]:
import nbformat
import os
import re

# ================= CONFIG =================

NOTEBOOK_MAP = {
    "main_1d_resnet_typ1.ipynb": "1d_resnet_typ1.ipynb",
    "main_1d_resnet_typ2.ipynb": "1d_resnet_typ2.ipynb",

}

FINAL_DIR = "result_and_plot_of_1d_resnet"
os.makedirs(FINAL_DIR, exist_ok=True)

MAX_EPOCH = 25
PATIENCE = 2

# =========================================


def one_line(s):
    """Make string Markdown-table safe and preserve key details"""
    if not s:
        return "—"
    cleaned = " ".join(s.replace("\n", " ").split())
    if len(cleaned) > 200:
        cleaned = cleaned[:197] + "..."
    return cleaned


def find_training_cell(nb):
    """Find the cell that contains the training loop (with output)"""
    for idx, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue
        
        # Check if this cell has training output
        has_output = False
        for out in cell.get("outputs", []):
            if out.output_type == "stream":
                text = out.get("text", "")
                if "Training Run Started:" in text or "Epoch [1/" in text:
                    has_output = True
                    break
        
        if has_output:
            return idx, cell.source
    
    return None, None


def extract_single_config(source, pattern):
    """Extract a single configuration line with multi-line support"""
    lines = source.splitlines()
    
    for i, line in enumerate(lines):
        # Look for the pattern with assignment
        if pattern in line and "=" in line:
            # Skip comments and variable assignments that aren't the config
            stripped = line.strip()
            if stripped.startswith("#"):
                continue
            
            # Found the line, now capture until balanced parentheses
            result = [line]
            balance = line.count("(") - line.count(")")
            
            # Simple assignment (no parentheses or balanced already)
            if balance <= 0:
                return line.strip()
            
            # Multi-line - need more lines
            j = i + 1
            while j < len(lines) and balance > 0:
                next_line = lines[j]
                result.append(next_line)
                balance += next_line.count("(") - next_line.count(")")
                j += 1
                
            return "\n".join(result).strip()
    
    return None


def extract_config(nb):
    """Extract configuration from the training loop cell itself"""
    found = {}
    
    # Find the training cell
    training_idx, training_source = find_training_cell(nb)
    
    if training_source is None:
        print("  ⚠️ Could not find training cell with output")
        return found
    
    print(f"  ✓ Found training cell at index {training_idx}")
    
    # Extract each config component from the training cell source
    model_cfg = extract_single_config(training_source, "model =")
    criterion_cfg = extract_single_config(training_source, "criterion =")
    optimizer_cfg = extract_single_config(training_source, "optimizer =")
    scheduler_cfg = extract_single_config(training_source, "scheduler =")
    
    if model_cfg:
        found["model ="] = model_cfg
    if criterion_cfg:
        found["criterion ="] = criterion_cfg
    if optimizer_cfg:
        found["optimizer ="] = optimizer_cfg
    if scheduler_cfg:
        found["scheduler ="] = scheduler_cfg
    
    print(f"  ✓ Extracted {len(found)}/4 config items")
    
    return found


def extract_training_log(nb):
    text = ""
    for cell in nb.cells:
        if cell.cell_type != "code":
            continue
        for out in cell.get("outputs", []):
            if out.output_type == "stream":
                text += out.get("text", "")
    return text


def parse_training(text):
    epoch_re = re.compile(
        r"Epoch \[(\d+)/\d+\] \| "
        r"Train Loss: ([\d.]+), Val Loss: ([\d.]+), "
        r"Train Acc: ([\d.]+)%, Val Acc: ([\d.]+)%, "
        r"LR: ([\d.eE+-]+)"
    )

    epochs = []
    for m in epoch_re.finditer(text):
        epochs.append({
            "epoch": int(m.group(1)),
            "train_loss": float(m.group(2)),
            "val_loss": float(m.group(3)),
            "train_acc": float(m.group(4)),
            "val_acc": float(m.group(5)),
            "lr": float(m.group(6)),
        })

    if not epochs:
        return None

    # Find epoch with best validation accuracy
    best = max(epochs, key=lambda x: x["val_acc"])

    stop_match = re.search(r"Early stopping at epoch (\d+)", text)
    stop_epoch = int(stop_match.group(1)) if stop_match else MAX_EPOCH

    # Extract best validation accuracy from the summary line
    best_val_match = re.search(r"Best Validation Accuracy:\s*([\d.]+)%", text)
    best_val_acc = float(best_val_match.group(1)) if best_val_match else best["val_acc"]

    test_match = re.search(r"Test Accuracy:\s*([\d.]+)%", text)
    test_acc = float(test_match.group(1)) if test_match else None

    # Extract training time
    time_match = re.search(r"Total Training Time:\s*([\d.]+)s", text)
    training_time = float(time_match.group(1)) if time_match else None

    return {
        "best_epoch": best["epoch"],
        "best_val_acc": best_val_acc,
        "best_train_acc": best["train_acc"],
        "train_loss_best": best["train_loss"],
        "val_loss_best": best["val_loss"],
        "init_lr": epochs[0]["lr"],
        "early_stop_epoch": stop_epoch,
        "test_acc": test_acc,
        "training_time": training_time,
    }


# ================= RUN =================

config_rows = []
result_rows = []

for src, dst in NOTEBOOK_MAP.items():
    if not os.path.exists(src):
        print(f"⚠️ Missing: {src}")
        continue

    nb = nbformat.read(src, as_version=4)
    name = dst.replace(".ipynb", "")

    print(f"\n📓 {src} -> {name}")
    
    cfg = extract_config(nb)
    log = extract_training_log(nb)
    stats = parse_training(log)
    
    # Debug output - show what was found
    for key in ["model =", "criterion =", "optimizer =", "scheduler ="]:
        if key in cfg:
            preview = cfg[key][:80].replace("\n", " ")
            print(f"  {key}: {preview}...")

    config_rows.append([
        src,  # Real notebook name
        name,  # Display name
        one_line(cfg.get("model =", "—")),
        one_line(cfg.get("criterion =", "—")),
        one_line(cfg.get("optimizer =", "—")),
        one_line(cfg.get("scheduler =", "—")),
        stats["init_lr"] if stats else "—",
    ])

    if stats:
        training_time_str = f"{stats['training_time']:.2f}s" if stats["training_time"] is not None else "—"
        result_rows.append([
            src,  # Real notebook name
            name,  # Display name
            MAX_EPOCH,
            "Yes",
            PATIENCE,
            stats["early_stop_epoch"],
            stats["best_epoch"],
            f"{stats['best_train_acc']:.2f}",
            f"{stats['best_val_acc']:.2f}",
            f"{stats['train_loss_best']:.4f}",
            f"{stats['val_loss_best']:.4f}",
            f"{stats['test_acc']:.2f}" if stats["test_acc"] is not None else "—",
            training_time_str,
        ])


# ================= WRITE MARKDOWN =================

md_path = os.path.join(FINAL_DIR, "summary.md")

with open(md_path, "w", encoding="utf-8") as f:

    f.write("# 🔧 Training Configuration\n\n")
    f.write("| Real Notebook | Display Name | Model | Criterion | Optimizer | Scheduler | Initial LR |\n")
    f.write("|---|---|---|---|---|---|---|\n")
    for r in config_rows:
        f.write("| " + " | ".join(map(str, r)) + " |\n")

    f.write("\n---\n\n")
    f.write("## 📊 Training Output Summary\n\n")
    f.write(
        "**Note:** Early stopping was enabled (patience = 2, max epoch = 25). "
        "All reported training and validation losses correspond to the epoch with the "
        "**highest validation accuracy**, not the final stopped epoch.\n\n"
    )

    f.write("| Real Notebook | Display Name | Max Epoch | Early Stop | Patience | Stop Epoch | Best Epoch | "
            "Best Train Acc (%) | Best Val Acc (%) | Train Loss | Val Loss | Test Acc (%) | Training Time |\n")
    f.write("|---|---|---|---|---|---|---|---|---|---|---|---|---|\n")

    for r in result_rows:
        f.write("| " + " | ".join(map(str, r)) + " |\n")

print(f"\n✅ DONE → {md_path}")


import pandas as pd
config_columns = [
    "Real Notebook", "Display Name", "Model",
    "Criterion", "Optimizer", "Scheduler", "Initial LR"
]

result_columns = [
    "Real Notebook", "Display Name", "Max Epoch", "Early Stop",
    "Patience", "Stop Epoch", "Best Epoch","Best Train Acc (%)",
    "Best Val Acc (%)", "Train Loss",
    "Val Loss", "Test Acc (%)", "Training Time"
]
df_config = pd.DataFrame(config_rows, columns=config_columns)
df_results = pd.DataFrame(result_rows, columns=result_columns)
excel_path = os.path.join(FINAL_DIR, "summary.xlsx")

with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    df_config.to_excel(writer, sheet_name="Training_Config", index=False)
    df_results.to_excel(writer, sheet_name="Training_Results", index=False)

print(f"📊 Excel summary saved → {excel_path}")



📓 main_1d_resnet_typ1.ipynb -> 1d_resnet_typ1
  ✓ Found training cell at index 37
  ✓ Extracted 4/4 config items
  model =: model = BenchmarkECGResNet1D(in_channels=12, num_classes=3).to(device)...
  criterion =: criterion = nn.CrossEntropyLoss()...
  optimizer =: optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)...
  scheduler =: scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(         optimizer, mode=...

📓 main_1d_resnet_typ2.ipynb -> 1d_resnet_typ2
  ✓ Found training cell at index 33
  ✓ Extracted 4/4 config items
  model =: model = BenchmarkECGResNet1D(in_channels=12, num_classes=3).to(device)...
  criterion =: criterion = nn.CrossEntropyLoss()...
  optimizer =: optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)...
  scheduler =: scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(         optimizer, mode=...

✅ DONE → result_and_plot_of_1d_resnet/summary.md
📊 Excel summary saved → result_and_plot_of_1d_resnet/summary